In [ ]:
import pandas as pd
import os

pd.options.display.float_format = '{:1.2f}'.format
pd.options.display.max_columns = None
pd.options.display.max_rows = None

In [ ]:
files = 0
for file in os.listdir("raw_listing"):
    files += 1
print(f"Number of listing files: {files}")

In [ ]:
df = pd.read_csv(f"raw_listing/CRMLSListing202502.csv", encoding='latin-1')

print('listing data:')
df.head()


In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
merged_df = None
folder_path = "raw_listing"
for file in os.listdir(folder_path):
    df = pd.read_csv(f"{folder_path}/{file}", encoding='latin-1')
    Residential_df = df[df['PropertyType'] == 'Residential']
    if merged_df is None:
        merged_df = df
        merged_residential_df = Residential_df
    else:
        merged_df = pd.concat([merged_df, df])
        merged_residential_df = pd.concat([merged_residential_df, Residential_df])

In [ ]:
merged_df.describe()

In [ ]:
merged_residential_df.describe()

In [ ]:
print("rows in merged df: " + str(len(merged_df)))
print("rows in merged residential df: " + str(len(merged_residential_df)))
print("difference in rows: " + str(len(merged_df) - len(merged_residential_df)))

In [ ]:
# merged_residential_df.to_csv("listing_merged_residential.csv", index=False)

Merged Analysis

In [ ]:
merged_df['PropertyType'].unique()

array(['Residential'], dtype=object)

In [ ]:
nulls = merged_df.isnull().sum()

total_rows = len(merged_df)

useless_Residential_columns = []

for column, null_count in nulls.items():
    percentage = (null_count / total_rows)
    if percentage > .9:
        print(f"{column}: {percentage:.2%} null")
        useless_Residential_columns.append(column)


FireplacesTotal: 100.00% null
AboveGradeFinishedArea: 100.00% null
TaxAnnualAmount: 100.00% null
BuilderName: 95.33% null
TaxYear: 100.00% null
BuildingAreaTotal: 91.11% null
ElementarySchoolDistrict: 100.00% null
CoBuyerAgentFirstName: 97.33% null
BelowGradeFinishedArea: 99.44% null
BusinessType: 100.00% null
CoveredSpaces: 100.00% null
LotSizeDimensions: 94.78% null
MiddleOrJuniorSchoolDistrict: 100.00% null


the columns with > 90% nulls were put into a list called "useless_Residential_columns"

In [ ]:
Price_LivingArea_DOM_df = merged_df[['ClosePrice', 'LivingArea', 'DaysOnMarket']]
Price_LivingArea_DOM_df.describe()

,ClosePrice,LivingArea,DaysOnMarket
count,145580.00,539627.00,540183.00
mean,1202136.39,1980.06,19.54
std,4292685.77,23382.69,26.77
min,525.00,0.00,-58.00
25%,600000.00,1247.00,5.00
50%,855000.00,1669.00,11.00
75%,1350000.00,2300.00,23.00
max,820000000.00,17021321.00,731.00


In [ ]:
# Price_LivingArea_DOM_df.to_csv("Price_LivingArea_DOM_listings.csv", index=False)

FRED Analysis

In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage = mortgage[mortgage['observation_date'] >= '2024-01-01'].reset_index().drop(columns=['index'])
mortgage.columns = ['date', 'rate_30yr_fixed']

In [ ]:
mortgage.head()

,date,rate_30yr_fixed
0,2024-01-04,6.62
1,2024-01-11,6.66
2,2024-01-18,6.60
3,2024-01-25,6.69
4,2024-02-01,6.63


In [ ]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
 mortgage.groupby('year_month')['rate_30yr_fixed']
 .mean()
 .reset_index()
)
mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,2024-01,6.64
1,2024-02,6.78
2,2024-03,6.82
3,2024-04,6.99
4,2024-05,7.06


In [ ]:
listings = merged_df
listings['year_month'] = pd.to_datetime(
    listings['ListingContractDate']
).dt.to_period('M')
listings[[ 'ListingContractDate','year_month']].head()

C:\Users\Jamesb\AppData\Local\Temp\ipykernel_23544\1923958767.py:1: DtypeWarning: Columns (2,43) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv(str(parent_dir)+'/week 1/listing_merged_residential.csv', encoding='latin-1')


,ListingContractDate,year_month
0,2024-01-01,2024-01
1,2024-01-24,2024-01
2,2024-01-12,2024-01
3,2024-01-20,2024-01
4,2024-01-12,2024-01


In [ ]:
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

In [ ]:
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

0
0


In [ ]:
listings_with_rates[
    ['ListingContractDate', 'year_month', 'ListPrice', 'rate_30yr_fixed']
].drop_duplicates('year_month').head(100)

,ListingContractDate,year_month,ListPrice,rate_30yr_fixed
0,2024-01-01,2024-01,1340000.00,6.64
17007,2024-02-18,2024-02,799999.00,6.78
34497,2024-03-27,2024-03,3550000.00,6.82
54998,2024-04-24,2024-04,549000.00,6.99
79023,2024-05-10,2024-05,929000.00,7.06
104470,2024-06-11,2024-06,699000.00,6.92
127780,2024-07-31,2024-07,949000.00,6.85
150799,2024-08-18,2024-08,300000.00,6.50
173014,2024-09-17,2024-09,570000.00,6.18
195271,2024-10-22,2024-10,788000.00,6.43


In [ ]:
# listings_with_rates.to_csv(str(parent_dir)+'/week 2-3/FRED_rates_listings.csv', index=False)

Data Cleaning